# Fine-tuning Qwen2.5-0.5B for ATIS structured-output (Colab, self-contained)

Teach a tiny model to turn an airline query into a fixed-schema JSON object.

**Schema** (8 keys, `null` when not mentioned):
```json
{"intent": ["flight"], "from_city": null, "to_city": null,
 "depart_date": null, "depart_time": null, "airline": null,
 "round_trip": null, "class_type": null}
```

This notebook is **self-contained**: it downloads ATIS, builds the JSON dataset, fine-tunes with
LoRA (TRL `SFTTrainer`), and evaluates before/after — no manual uploads.

All randomness is seeded (`SEED`) so the split and training are reproducible across runs.

> **Runtime:** set `Runtime -> Change runtime type -> GPU` (a T4 is enough).

## 1. Install dependencies (Colab uses pip)

In [ ]:
!pip install -q -U transformers trl peft datasets accelerate bitsandbytes

## 2. Config + schema

In [ ]:
import json, urllib.request
import torch
from transformers import set_seed

SEED = 42
set_seed(SEED)  # seed python / numpy / torch for reproducible runs

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
TEST_FRACTION = 0.10

INTENT_ORDER = ["flight", "airfare", "ground_service", "airline"]
CHOSEN = set(INTENT_ORDER)
SCHEMA_FIELDS = ["from_city", "to_city", "depart_date", "depart_time",
                 "airline", "round_trip", "class_type"]
FIELD_MAP = {
    "fromloc": "from_city", "toloc": "to_city",
    "depart_date": "depart_date", "depart_time": "depart_time",
    "airline_name": "airline", "airline_code": "airline",
    "round_trip": "round_trip", "class_type": "class_type",
}

SYSTEM_PROMPT = (
    "You extract structured information from airline-travel queries. "
    "Respond with ONLY a JSON object with exactly these keys: "
    'intent, from_city, to_city, depart_date, depart_time, airline, round_trip, class_type. '
    "'intent' is a list whose values are a subset of "
    '["flight", "airfare", "ground_service", "airline"]. '
    "Every other field is a string copied from the query, or null if not mentioned. "
    "Output only the JSON, no extra text."
)

## 3. Download ATIS, build JSON records, and make a fixed 90/10 split

Same transform as the local notebook. We then pool ATIS's predefined train/test files and make
a single **seeded** 90/10 split, so the partition is identical on every run (comparable results).

In [ ]:
BASE = "https://huggingface.co/datasets/tuetschek/atis/resolve/main"
for split in ["train", "test"]:
    urllib.request.urlretrieve(f"{BASE}/atis_{split}.csv", f"atis_{split}.csv")

import csv, random

def read_csv(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f))

def normalize_intent(intent_str):
    parts = set(intent_str.split("+"))
    if not parts <= CHOSEN:
        return None
    return [i for i in INTENT_ORDER if i in parts]

def extract_spans(text, slots):
    spans, base, cur = [], None, []
    for tok, tag in zip(text.split(), slots.split()):
        if tag == "O":
            if base is not None:
                spans.append((base, " ".join(cur))); base, cur = None, []
            continue
        prefix, b = tag[0], tag[2:]
        if prefix == "B" or b != base:
            if base is not None:
                spans.append((base, " ".join(cur)))
            base, cur = b, [tok]
        else:
            cur.append(tok)
    if base is not None:
        spans.append((base, " ".join(cur)))
    return spans

def to_record(text, slots):
    rec = {f: None for f in SCHEMA_FIELDS}
    for b, span_text in extract_spans(text, slots):
        field = FIELD_MAP.get(b.split(".")[0])
        if field is not None and rec[field] is None:
            rec[field] = span_text
    return rec

def build(path):
    rows = []
    for r in read_csv(path):
        intent = normalize_intent(r["intent"])
        if intent is None:
            continue
        rows.append({"query": r["text"],
                     "output": {"intent": intent, **to_record(r["text"], r["slots"])}})
    return rows

# Pool both predefined splits, then make a fixed seeded 90/10 split.
all_rows = build("atis_train.csv") + build("atis_test.csv")
random.Random(SEED).shuffle(all_rows)
n_test = round(len(all_rows) * TEST_FRACTION)
test_rows = all_rows[:n_test]
train_rows = all_rows[n_test:]
print(f"total: {len(all_rows)} | train: {len(train_rows)} | test: {len(test_rows)} "
      f"({len(test_rows)/len(all_rows):.1%} test)")
print("example:", json.dumps(train_rows[0], indent=2))

## 4. Format as prompt/completion pairs

Conversational **prompt** (system + user) and **completion** (assistant JSON). TRL applies the
chat template and trains on the completion only (the prompt is masked out of the loss).

In [ ]:
from datasets import Dataset

def to_prompt(query):
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": query}]

def dump(record):
    return json.dumps(record, ensure_ascii=False)

def to_example(row):
    return {"prompt": to_prompt(row["query"]),
            "completion": [{"role": "assistant", "content": dump(row["output"])}]}

train_ds = Dataset.from_list([to_example(r) for r in train_rows])
test_ds = Dataset.from_list([to_example(r) for r in test_rows])
print(train_ds)
print("\ncompletion[0]:", train_ds[0]["completion"][0]["content"])

## 5. Load model + tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
dtype = torch.bfloat16 if bf16 else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype, device_map="auto")
print(f"loaded {MODEL_NAME} | bf16={bf16} | device={model.device}")

## 6. Generation + evaluation helpers

In [ ]:
@torch.no_grad()
def generate(queries, max_new_tokens=128, batch_size=16):
    """Greedy-decode JSON strings for a list of queries (batched, left-padded)."""
    model.eval()
    prev_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    out = []
    for i in range(0, len(queries), batch_size):
        batch = queries[i:i + batch_size]
        prompts = [tokenizer.apply_chat_template(to_prompt(q), tokenize=False,
                                                 add_generation_prompt=True) for q in batch]
        enc = tokenizer(prompts, return_tensors="pt", padding=True,
                        truncation=True, max_length=384).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
        for j in range(len(batch)):
            new = gen[j][enc["input_ids"].shape[1]:]
            out.append(tokenizer.decode(new, skip_special_tokens=True))
    tokenizer.padding_side = prev_side
    return out

def parse_json(text):
    s, e = text.find("{"), text.rfind("}")
    if s == -1 or e == -1 or e < s:
        return None
    try:
        return json.loads(text[s:e + 1])
    except json.JSONDecodeError:
        return None

def _norm(v):
    return v.strip().lower() if isinstance(v, str) else v

def evaluate(rows, max_new_tokens=128):
    preds = generate([r["query"] for r in rows], max_new_tokens=max_new_tokens)
    n = len(rows)
    valid = exact = intent_ok = 0
    field_ok = {f: 0 for f in SCHEMA_FIELDS}
    for row, text in zip(rows, preds):
        pred = parse_json(text)
        if pred is None:
            continue
        valid += 1
        gold = row["output"]
        gi = set(gold["intent"])
        pi = set(pred["intent"]) if isinstance(pred.get("intent"), list) else set()
        all_ok = gi == pi
        if all_ok:
            intent_ok += 1
        for f in SCHEMA_FIELDS:
            if _norm(pred.get(f)) == _norm(gold[f]):
                field_ok[f] += 1
            else:
                all_ok = False
        exact += all_ok
    print(f"n={n}")
    print(f"valid JSON   : {valid/n:6.1%}")
    print(f"exact match  : {exact/n:6.1%}")
    print(f"intent acc   : {intent_ok/n:6.1%}")
    for f in SCHEMA_FIELDS:
        print(f"  {f:<12}: {field_ok[f]/n:6.1%}")

def show(rows, idxs):
    preds = generate([rows[i]["query"] for i in idxs])
    for i, text in zip(idxs, preds):
        print("Q   :", rows[i]["query"])
        print("gold:", dump(rows[i]["output"]))
        print("pred:", text.strip())
        print("-" * 70)

## 7. Baseline (before fine-tuning)

In [ ]:
DEMO = [0, 5, 17, 42]
print("=== BEFORE: sample generations ===")
show(test_rows, DEMO)
print("\n=== BEFORE: metrics on the full test set ===")
evaluate(test_rows)

## 8. Fine-tune with LoRA

In [ ]:
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

tokenizer.padding_side = "right"  # training

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules="all-linear", task_type="CAUSAL_LM",
)

sft_config = SFTConfig(
    output_dir="atis-qwen-lora",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=25,
    max_length=384,
    packing=False,
    bf16=bf16, fp16=not bf16,
    report_to="none",
    save_strategy="no",
    seed=SEED,
    data_seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    peft_config=peft_config,
    processing_class=tokenizer,
)
trainer.train()

## 9. After fine-tuning

In [ ]:
print("=== AFTER: same sample generations ===")
show(test_rows, DEMO)
print("\n=== AFTER: metrics on the full test set ===")
evaluate(test_rows)

## 10. Save the adapter
Saves the LoRA adapter (small). Download it, or `merge_and_unload()` to export a full model.

In [ ]:
trainer.save_model("atis-qwen-lora")
tokenizer.save_pretrained("atis-qwen-lora")
print("saved adapter to ./atis-qwen-lora")
# Optional: zip for download
# !zip -rq atis-qwen-lora.zip atis-qwen-lora && print('zipped')

## Next steps
See `reference/design-decisions.md` (§6) — multi-value fields, the canonicalization
experiment (typed/resolved values), structured-output enforcement, broader intent/slot
coverage, and a model/LoRA-rank sweep.